## MMTHE01 - Masters Thesis

### C. Thesis - Create a working model - Original Sampled Data
* Splitting the data into train and test data
* Feature scaling
* Model training

#### Importing the libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
import time
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from xgboost import XGBClassifier

from sklearn.preprocessing import LabelEncoder

In [2]:
### Import libraries to save models
import pickle
from tensorflow.keras.models import Model, load_model

In [3]:
# Check your current working directory
cwd = os.getcwd()

# Define your relative path
relative_path = r"Analysis"  # adjust this relative to cwd

# Build the full path
full_path = os.path.join(cwd, relative_path)

# Check if it exists before changing
if os.path.exists(full_path):
    os.chdir(full_path)
    print("Changed directory to:", full_path)
else:
    print("Folder does not exist:", full_path)

Changed directory to: C:\Users\eaber\Documents\Masters Thesis Final\Analysis


#### Load Dataset

In [4]:
with open("saved_data/final_dataset.pkl","rb") as f:
    dataset = pickle.load(f)

In [5]:
dataset.head()

,acc_open_past_24mths,annual_inc,avg_cur_bal,bc_open_to_buy,bc_util,collections_12_mths_ex_med,delinq_2yrs,delinq_amnt,dti,earliest_cr_line_month_Apr,...,tot_hi_cred_lim,total_acc,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,total_rev_hi_lim,verification_status_Not Verified,verification_status_Source Verified,verification_status_Verified,default_status
0,4.517166,33000.0,13533.888648,11418.697365,57.908076,0.000000,0.0,0.0,2.15,1.0,...,178248.102764,6.0,51080.557557,23226.657827,43762.092547,34606.178028,1.0,0.0,0.0,0
1,4.517166,12000.0,13533.888648,11418.697365,57.908076,0.018288,0.0,0.0,4.40,0.0,...,178248.102764,16.0,51080.557557,23226.657827,43762.092547,34606.178028,1.0,0.0,0.0,0
2,4.517166,59000.0,13533.888648,11418.697365,57.908076,0.000000,0.0,0.0,23.86,0.0,...,178248.102764,50.0,51080.557557,23226.657827,43762.092547,34606.178028,1.0,0.0,0.0,0
3,4.517166,98000.0,13533.888648,11418.697365,57.908076,0.000000,0.0,0.0,12.53,0.0,...,178248.102764,12.0,51080.557557,23226.657827,43762.092547,34606.178028,1.0,0.0,0.0,0
4,4.517166,7280.0,13533.888648,11418.697365,57.908076,0.018288,0.0,0.0,17.31,0.0,...,178248.102764,4.0,51080.557557,23226.657827,43762.092547,34606.178028,1.0,0.0,0.0,0


In [6]:
dataset.shape

(226067, 110)

In [7]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226067 entries, 0 to 226066
Columns: 110 entries, acc_open_past_24mths to default_status
dtypes: category(1), float64(104), int64(3), object(2)
memory usage: 188.2+ MB


#### Split the data into Train-Test

In [8]:
X = dataset.drop(columns=['default_status']).values
y = dataset['default_status'].values

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify=y, random_state = 1)

#### Feature Scaling

In [10]:
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

#### Loading the models

In [11]:
# load Random Forest model
with open("saved_models/random_forest_model_original.pkl", "rb") as f:
    rf = pickle.load(f)

In [12]:
# load XGBoost model
xgb = xgb.XGBClassifier()
xgb.load_model("saved_models/xgb_model_original.json")

In [13]:
# load ANN model
model = load_model("saved_models/ann_model_original.keras")

In [14]:
# load SVM model
with open("saved_models/svm_pipeline_original.pkl", "rb") as f:
    pipeline = pickle.load(f)

In [15]:
# load GBM model
with open("saved_models/gbm_model_original.pkl", "rb") as f:
    gbm = pickle.load(f)

### 3.6 Cross Validation Checks for the Supervised Learning Models

In [16]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, f1_score

In [17]:
# Define multiple scoring metrics
scoring = {
    'auc': 'roc_auc',
    'recall': 'recall',
    'precision': 'precision',
    'F1': make_scorer(f1_score, average='macro')
}

#### 3.6.1 Cross Validation for the Random Forest Model

In [18]:
# Run cross-validation once
cv_results_rf = cross_validate(
    estimator=rf,
    X=X_train,
    y=y_train,
    cv=10,
    scoring=scoring
)

In [19]:
# Print results - RF AUC
print("AUC Random Forest: {:.3f} %".format(cv_results_rf['test_auc'].mean() * 100))
print("AUC Standard Deviation Random Forest: {:.5f}".format(cv_results_rf['test_auc'].std()))

AUC Random Forest: 76.120 %
AUC Standard Deviation Random Forest: 0.00555


In [20]:
# Print results - RF Recall
print("Recall Random Forest: {:.4f}".format(cv_results_rf['test_recall'].mean()))
print("Recall Standard Deviation Random Forest: {:.5f}".format(cv_results_rf['test_recall'].std()))

Recall Random Forest: 0.6958
Recall Standard Deviation Random Forest: 0.01483


In [21]:
# Print results - RF precision
print("Precision Random Forest: {:.4f}".format(cv_results_rf['test_precision'].mean()))
print("Precision Standard Deviation Random Forest: {:.5f}".format(cv_results_rf['test_precision'].std()))

Precision Random Forest: 0.2235
Precision Standard Deviation Random Forest: 0.00339


In [22]:
# Print results - Random Forest F1 Score
print("F1-score Random Forest: {:.4f}".format(cv_results_rf['test_F1'].mean()))
print("F1-score Standard Deviation Random Forest: {:.5f}".format(cv_results_rf['test_F1'].std()))

F1-score Random Forest: 0.5658
F1-score Standard Deviation Random Forest: 0.00322


#### 3.6.2 Cross Validation for the XGBoost Model

In [23]:
# Run cross-validation once
cv_results_xgb = cross_validate(
    estimator=xgb,
    X=X_train,
    y=y_train,
    cv=10,
    scoring=scoring
)

In [24]:
# Print results - XGB AUC
print("AUC XGBoost: {:.3f} %".format(cv_results_xgb['test_auc'].mean() * 100))
print("AUC Standard Deviation XGBoost: {:.5f}".format(cv_results_xgb['test_auc'].std()))

AUC XGBoost: 76.694 %
AUC Standard Deviation XGBoost: 0.00683


In [25]:
# Print results - XGB Recall
print("Recall XGBoost: {:.4f}".format(cv_results_xgb['test_recall'].mean()))
print("Recall Standard Deviation XGBoost: {:.5f}".format(cv_results_xgb['test_recall'].std()))

Recall XGBoost: 0.0375
Recall Standard Deviation XGBoost: 0.00429


In [26]:
# Print results - XGB Precision
print("Precision XGBoost: {:.4f}".format(cv_results_xgb['test_precision'].mean()))
print("Precision Standard Deviation XGBoost: {:.5f}".format(cv_results_xgb['test_precision'].std()))

Precision XGBoost: 0.4035
Precision Standard Deviation XGBoost: 0.02988


In [27]:
# Print results - XGB F1 Score
print("F1-score XGBoost: {:.4f}".format(cv_results_xgb['test_F1'].mean()))
print("F1-score Standard Deviation XGBoost: {:.5f}".format(cv_results_xgb['test_F1'].std()))

F1-score XGBoost: 0.5029
F1-score Standard Deviation XGBoost: 0.00385


#### 3.6.3 Cross Validation for the ANN Model

In [28]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import KFold

# --------------------------
# 1. Define your ANN model builder
# --------------------------
def create_ann_model(input_dim):
    # Build an ANN model
    model = Sequential([
        Input(shape=(X_train.shape[1],)),  # Explicit Input layer instead of input_dim in Dense
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    
    # Early stopping to prevent overfitting
    early_stop = EarlyStopping(monitor='val_AUC', patience=3, restore_best_weights=True, mode='max')

    return model

In [29]:
# --------------------------
# 2. Manual cross-validation loop
# --------------------------
def cross_validate_ann(X, y):

    kf = KFold(n_splits=10, shuffle=True, random_state=1)

    auc_scores = []
    recall_scores = []
    precision_scores = []
    f1_scores = []

    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]


        classes = np.unique(y_train)
        class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
        class_weights = {i: class_weights[i] for i in range(len(classes))}

        # Build model fresh each fold
        model = create_ann_model(input_dim=X.shape[1])

        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])

        # Early stopping to prevent overfitting
        early_stop = EarlyStopping(monitor='val_AUC', patience=3, restore_best_weights=True, mode='max')

        # Train
            # Train the ANN model with timing
        history = model.fit(
            X_train, y_train,
            validation_split=0.2,
            epochs=50,
            batch_size=256,
            callbacks=[early_stop],
            class_weight=class_weights,
            verbose=1
        )
    

        # Predict probabilities for AUC
        y_prob = model.predict(X_test).ravel()

        # Convert to class labels
        y_pred = (y_prob >= 0.6).astype(int)

        # Compute metrics
        auc_scores.append(roc_auc_score(y_test, y_prob))
        recall_scores.append(recall_score(y_test, y_pred))
        precision_scores.append(precision_score(y_test, y_pred))
        f1_scores.append(f1_score(y_test, y_pred, average='macro'))

    # Return results as a dictionary similar to sklearn
    return {
        "test_auc": np.array(auc_scores),
        "test_recall": np.array(recall_scores),
        "test_precision": np.array(precision_scores),
        "test_F1": np.array(f1_scores)
    }

In [30]:
cv_results_ann = cross_validate_ann(X_train_scaled, y_train)

Epoch 1/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - AUC: 0.6492 - loss: 0.6742 - val_AUC: 0.7407 - val_loss: 0.5951
Epoch 2/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - AUC: 0.7302 - loss: 0.6039 - val_AUC: 0.7550 - val_loss: 0.5839
Epoch 3/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - AUC: 0.7462 - loss: 0.5938 - val_AUC: 0.7609 - val_loss: 0.5625
Epoch 4/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - AUC: 0.7566 - loss: 0.5837 - val_AUC: 0.7655 - val_loss: 0.5695
Epoch 5/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - AUC: 0.7646 - loss: 0.5714 - val_AUC: 0.7683 - val_loss: 0.5635
Epoch 6/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - AUC: 0.7636 - loss: 0.5677 - val_AUC: 0.7692 - val_loss: 0.5611
Epoch 7/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - AUC: 0.7700 - loss: 0.5703 - val_AUC: 0.7703 - val_loss: 0.5643
Epoch 8/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - AUC: 0.7678 - loss: 0.5690 - val_AUC: 0.7715 - val_loss: 0.5556
Epoch 9/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - AU

In [31]:
# Print results - ANN AUC
print("AUC ANN: {:.3f} %".format(cv_results_ann['test_auc'].mean() * 100))
print("AUC Standard Deviation ANN: {:.5f}".format(cv_results_ann['test_auc'].std()))

AUC ANN: 76.981 %
AUC Standard Deviation ANN: 0.00427


In [32]:
# Print results - ANN Recall
print("Recall ANN: {:.4f}".format(cv_results_ann['test_recall'].mean()))
print("Recall Standard Deviation ANN: {:.5f}".format(cv_results_ann['test_recall'].std()))

Recall ANN: 0.5402
Recall Standard Deviation ANN: 0.02532


In [33]:
# Print results - ANN Precision
print("Precision ANN: {:.4f}".format(cv_results_ann['test_precision'].mean()))
print("Precision Standard Deviation ANN: {:.5f}".format(cv_results_ann['test_precision'].std()))

Precision ANN: 0.2670
Precision Standard Deviation ANN: 0.00901


In [34]:
# Print results - ANN F1 Score
print("F1-score ANN: {:.4f}".format(cv_results_ann['test_F1'].mean()))
print("F1-score Standard Deviation ANN: {:.5f}".format(cv_results_ann['test_F1'].std()))

F1-score ANN: 0.6102
F1-score Standard Deviation ANN: 0.00563


#### 3.6.4 Cross Validation for the SVM Model

In [35]:
# Run cross-validation
cv_results_svm = cross_validate(
    estimator=pipeline,
    X=X_train_scaled,
    y=y_train,
    cv=10,
    scoring=scoring
)

In [36]:
# Print results - SVM AUC
print("AUC SVM: {:.3f} %".format(cv_results_svm['test_auc'].mean() * 100))
print("AUC Standard Deviation SVM: {:.5f}".format(cv_results_svm['test_auc'].std()))

AUC SVM: 74.725 %
AUC Standard Deviation SVM: 0.00706


In [37]:
# Print results - SVM Recall
print("Recall SVM: {:.4f}".format(cv_results_svm['test_recall'].mean()))
print("Recall Standard Deviation SVM: {:.5f}".format(cv_results_svm['test_recall'].std()))

Recall SVM: 0.6821
Recall Standard Deviation SVM: 0.01566


In [38]:
# Print results - SVM Precision
print("Precision SVM: {:.4f}".format(cv_results_svm['test_precision'].mean()))
print("Precision Standard Deviation SVM: {:.5f}".format(cv_results_svm['test_precision'].std()))

Precision SVM: 0.2181
Precision Standard Deviation SVM: 0.00485


In [39]:
# Print results - SVM F1 Score
print("F1-score SVM: {:.4f}".format(cv_results_svm['test_F1'].mean()))
print("F1-score Standard Deviation SVM: {:.5f}".format(cv_results_svm['test_F1'].std()))

F1-score SVM: 0.5602
F1-score Standard Deviation SVM: 0.00497


#### 3.6.5 Cross Validation for the GBM Model

In [40]:
# Run cross-validation
cv_results_gbm = cross_validate(
    estimator=gbm,
    X=X_train,
    y=y_train,
    cv=10,
    scoring=scoring
)

In [41]:
# Print results - GBM AUC
print("AUC GBM: {:.3f} %".format(cv_results_gbm['test_auc'].mean() * 100))
print("AUC Standard Deviation GBM: {:.5f}".format(cv_results_gbm['test_auc'].std()))

AUC GBM: 77.283 %
AUC Standard Deviation GBM: 0.00628


In [42]:
# Print results - GBM Recall
print("Recall GBM: {:.4f}".format(cv_results_gbm['test_recall'].mean()))
print("Recall Standard Deviation GBM: {:.5f}".format(cv_results_gbm['test_recall'].std()))

Recall GBM: 0.0220
Recall Standard Deviation GBM: 0.00246


In [43]:
# Print results - GBM Precision
print("Precision GBM: {:.4f}".format(cv_results_gbm['test_precision'].mean()))
print("Precision Standard Deviation GBM: {:.5f}".format(cv_results_gbm['test_precision'].std()))

Precision GBM: 0.4760
Precision Standard Deviation GBM: 0.04861


In [44]:
# Print results - GBM F1 Score
print("F1-score GBM: {:.4f}".format(cv_results_gbm['test_F1'].mean()))
print("F1-score Standard Deviation GBM: {:.5f}".format(cv_results_gbm['test_F1'].std()))

F1-score GBM: 0.4902
F1-score Standard Deviation GBM: 0.00239
